In [ ]:
!pip -q install pandas pyarrow

import pandas as pd

base = "https://us.data.motherduck.com/job_postings/parquet"
urls = []
for y in [2025, 2026]:
    for m in range(1, 13):
        urls.append(f"{base}/year={y}/month={m:02d}/jobs.parquet")

frames = []
for url in urls:
    try:
        frames.append(pd.read_parquet(url))
        print("Loaded:", url)
    except Exception as e:
        print("Skipped:", url)

df = pd.concat(frames, ignore_index=True)
print("\nTotal rows:", len(df))

In [ ]:
import re

df = df.dropna(subset=["description", "title"]).copy()
df["description"] = df["description"].astype(str)
df = df[df["description"].str.len() >= 100].copy()

def clean_text(text):
    text = str(text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["clean_description"] = df["description"].apply(clean_text)

def assign_category(title):
    t = str(title).lower()
    if any(k in t for k in ["data engineer", "etl", "databricks", "data pipeline", "big data"]):
        return "Data Engineering"
    if any(k in t for k in ["data scientist", "machine learning", "ml engineer", "ai engineer", "deep learning"]):
        return "Data Science/ML"
    if any(k in t for k in ["business intelligence", "bi ", "bi developer", "bi analyst", "tableau", "power bi"]):
        return "Business Intelligence"
    if any(k in t for k in ["data analyst", "analytics", "analyst"]):
        return "Analytics"
    return "Other"

df["category"] = df["title"].apply(assign_category)
df = df[df["category"] != "Other"].copy()

print("Final rows:", len(df))
print(df["category"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42
X = df["clean_description"].values
y = df["category"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print("Train:", len(X_train), "| Test:", len(X_test))

# Store TF-IDF results from your previous run so we have the final table ready
results = [{
    "Model": "TF-IDF + LR",
    "Accuracy": 0.9155,
    "Macro Precision": 0.8416,
    "Macro Recall": 0.9177,
    "Macro F1": 0.8726,
    "Train Time (s)": 68.61,
    "Inference Time (s)": 0.0791,
    "Parameters": "—"
}]

In [ ]:
!pip -q install transformers torch

import torch
import time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)
NUM_LABELS = len(le.classes_)

MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 256
BATCH_SIZE = 16
EPOCHS     = 1
LR         = 2e-5

# use_fast=True speeds up tokenization and saves RAM
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# --- NEW MEMORY-SAFE DATASET ---
# Instead of processing all 150k rows at once, we process them only when the DataLoader asks for them


In [ ]:
class SafeJobDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        # Convert to tensor ONLY when requested by the batch loader
        enc = tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=MAX_LEN, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = SafeJobDataset(X_train, y_train_enc)
test_ds  = SafeJobDataset(X_test,  y_test_enc)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Clear RAM before loading model


In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)
optimizer = AdamW(model.parameters(), lr=LR)



In [ ]:
model.train()
t_start = time.perf_counter()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | avg loss: {total_loss/len(train_loader):.4f}")
bert_train_time = time.perf_counter() - t_start

In [ ]:
model.eval()
preds = []
t_start = time.perf_counter()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        logits = model(**batch).logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
bert_inference_time = time.perf_counter() - t_start

y_pred_bert = le.inverse_transform(np.array(preds))

acc  = accuracy_score(y_test, y_pred_bert)
prec = precision_score(y_test, y_pred_bert, average="macro", zero_division=0)
rec  = recall_score(y_test, y_pred_bert, average="macro", zero_division=0)
f1   = f1_score(y_test, y_pred_bert, average="macro", zero_division=0)

print("\n" + "="*60)
print("BERT RESULTS")
print("="*60)
print(f"Accuracy        : {acc:.4f}")
print(f"Macro Precision : {prec:.4f}")
print(f"Macro Recall    : {rec:.4f}")
print(f"Macro F1        : {f1:.4f}")
print(f"Training time   : {bert_train_time:.2f} s")
print(f"Inference time  : {bert_inference_time:.4f} s")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_bert, zero_division=0))

results.append({
    "Model": "BERT",
    "Accuracy": acc,
    "Macro Precision": prec,
    "Macro Recall": rec,
    "Macro F1": f1,
    "Train Time (s)": bert_train_time,
    "Inference Time (s)": bert_inference_time,
    "Parameters": sum(p.numel() for p in model.parameters())
})

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import time

print("Training TF-IDF + Logistic Regression...")
t_start = time.perf_counter()

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english", min_df=3)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_tfidf, y_train)

y_pred_tfidf = lr.predict(X_test_tfidf)
tfidf_time = time.perf_counter() - t_start

print("\n" + "="*60)
print("TF-IDF + LOGISTIC REGRESSION RESULTS")
print("="*60)
print(f"Accuracy        : {accuracy_score(y_test, y_pred_tfidf):.4f}")
print(f"Macro Precision : {precision_score(y_test, y_pred_tfidf, average='macro', zero_division=0):.4f}")
print(f"Macro Recall    : {recall_score(y_test, y_pred_tfidf, average='macro', zero_division=0):.4f}")
print(f"Macro F1        : {f1_score(y_test, y_pred_tfidf, average='macro', zero_division=0):.4f}")
print(f"Total time      : {tfidf_time:.2f} s")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_tfidf, zero_division=0))

In [ ]:
import gc
import torch
import time
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# 1. Redefine the SafeJobDataset class
class SafeJobDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=MAX_LEN, return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Clear RAM from previous models
gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# 2. Re-create datasets with DistilBERT tokenizer
train_ds = SafeJobDataset(X_train, y_train_enc)
test_ds  = SafeJobDataset(X_test,  y_test_enc)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Load Model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)
optimizer = AdamW(model.parameters(), lr=LR)

# Training
print("Training DistilBERT...")
model.train()
t_start = time.perf_counter()
for epoch in range(EPOCHS): # This is set to 1
    total_loss = 0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(**batch)
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | avg loss: {total_loss/len(train_loader):.4f}")
distil_train_time = time.perf_counter() - t_start

# Inference
model.eval()
preds = []
t_start = time.perf_counter()
with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
        logits = model(**batch).logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
distil_inference_time = time.perf_counter() - t_start

y_pred_distil = le.inverse_transform(np.array(preds))

acc  = accuracy_score(y_test, y_pred_distil)
prec = precision_score(y_test, y_pred_distil, average="macro", zero_division=0)
rec  = recall_score(y_test, y_pred_distil, average="macro", zero_division=0)
f1   = f1_score(y_test, y_pred_distil, average="macro", zero_division=0)

print("\n" + "="*60)
print("DISTILBERT RESULTS")
print("="*60)
print(f"Accuracy        : {acc:.4f}")
print(f"Macro Precision : {prec:.4f}")
print(f"Macro Recall    : {rec:.4f}")
print(f"Macro F1        : {f1:.4f}")
print(f"Training time   : {distil_train_time:.2f} s")
print(f"Inference time  : {distil_inference_time:.4f} s")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_distil, zero_division=0))

# Save DistilBERT to results
results.append({
    "Model": "DistilBERT",
    "Accuracy": acc,
    "Macro Precision": prec,
    "Macro Recall": rec,
    "Macro F1": f1,
    "Train Time (s)": distil_train_time,
    "Inference Time (s)": distil_inference_time,
    "Parameters": sum(p.numel() for p in model.parameters())
})

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import torch
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# 1. FINAL COMPARISON TABLE & CHARTS
# ==========================================
results = [
    {"Model": "TF-IDF + LR", "Accuracy": 0.9153, "Macro F1": 0.8722, "Train Time (s)": 240.98, "Inference Time (s)": 0.08},
    {"Model": "BERT", "Accuracy": 0.9271, "Macro F1": 0.8917, "Train Time (s)": 6369.44, "Inference Time (s)": 559.83},
    {"Model": "DistilBERT", "Accuracy": 0.9250, "Macro F1": 0.8894, "Train Time (s)": 3282.73, "Inference Time (s)": 313.44}
]
df_results = pd.DataFrame(results)

print("\n" + "="*80)
print("FINAL COMPARATIVE ANALYSIS")
print("="*80)
print(df_results.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else x))

# Plot Accuracy and F1
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
x = np.arange(len(df_results["Model"]))
width = 0.35

ax[0].bar(x - width/2, df_results["Accuracy"], width, label='Accuracy', color='steelblue')
ax[0].bar(x + width/2, df_results["Macro F1"], width, label='Macro F1', color='orange')
ax[0].set_xticks(x)
ax[0].set_xticklabels(df_results["Model"])
ax[0].set_title("Accuracy vs Macro F1")
ax[0].set_ylim(0.8, 1.0)
ax[0].legend()

# Plot Inference Time
ax[1].bar(df_results["Model"], df_results["Inference Time (s)"], color='teal')
ax[1].set_title("Inference Time (Lower is Better)")
ax[1].set_ylabel("Seconds")
plt.tight_layout()
plt.show()

# ==========================================
# 2. SEMANTIC JOB MATCHING DEMO
# ==========================================
print("\n" + "="*80)
print("SEMANTIC JOB MATCHING DEMO (Using BERT)")
print("="*80)

# Create a fake candidate profile
candidate_profile = "Highly motivated Data Scientist with 3 years of experience in Python, Machine Learning, deep learning, and building predictive models."
print(f"Candidate Profile: '{candidate_profile}'\n")

# Use the BERT tokenizer and model to encode the candidate
model.eval()
inputs = tokenizer(candidate_profile, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)

with torch.no_grad():
    # Get BERT's [CLS] token representation for the candidate
    outputs = model.distilbert(**inputs) # using distilbert base for demo
    candidate_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()

# Sample 100 random jobs from the test set to act as our "job board"
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), 100, replace=False)
job_descriptions = X_test[sample_idx]

job_embs = []
with torch.no_grad():
    for text in job_descriptions:
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=MAX_LEN).to(device)
        outputs = model.distilbert(**inputs)
        job_embs.append(outputs.last_hidden_state[:, 0, :].cpu().numpy())
job_embs = np.vstack(job_embs)

# Calculate Cosine Similarity
similarities = cosine_similarity(candidate_emb, job_embs)[0]
top_3_idx = similarities.argsort()[-3:][::-1]

print("Top 3 Recommended Jobs for this candidate:\n")
for rank, idx in enumerate(top_3_idx, 1):
    score = similarities[idx]
    snippet = job_descriptions[idx][:250].replace('\n', ' ')
    print(f"Rank {rank} | Similarity Score: {score:.4f}")
    print(f"Job Snippet: {snippet}...\n")